# Covenant monitoring

**Maintenance** covenants are tested on a schedule (often quarterly); **incurrence** tests gate specific actions. **Springing** covenants activate when a revolver is drawn. Here we **project leverage** and compare it to a threshold under scenarios.

## Concept

The statement model computes **Net Debt / EBITDA** as a tested metric. A typed `CovenantEngine` built from the `cov_lite` template applies the covenant package to base and downside snapshots and returns Rust-owned pass/fail and headroom results (`CovenantReport`), including support for richer cure, basket, waiver, and step-down policies.

In [ ]:
from datetime import date

from _shared import series
from finstack_quant.core.market_data import MarketContext
from finstack_quant.covenants import CovenantEngine, CovenantForecastConfig, CovenantTemplate, covenant_template, reports_table
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import (
    DependencyTracer, StatementCovenantSpec, assess_statement_covenants, forecast_statement_covenants,
)

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2"]
AS_OF_DATES = ["2025-03-31", "2025-06-30", "2025-09-30", "2025-12-31", "2026-03-31", "2026-06-30"]

b = ModelBuilder("covenant-demo")
b.periods("2025Q1..2026Q2", "2025Q2")
b.value("revenue", series(PERIODS, [100.0, 102.0, 105.0, 108.0, 110.0, 112.0]))
b.compute("ebitda", "revenue * 0.2")
b.value("total_debt", series(PERIODS, [95.0, 93.0, 91.0, 89.0, 87.0, 85.0]))
b.value("cash", series(PERIODS, [10.0, 10.5, 11.0, 11.5, 12.0, 12.5]))
b.compute("net_debt", "total_debt - cash")
b.compute("leverage", "if(ebitda > 0, net_debt / ebitda, 999.9)")

cutoff = date(2025, 7, 31)
b.source_snapshot("covenant-demo-july-2025")
for node in ("revenue", "total_debt", "cash"):
    b.availability_dates(node, [("2025Q1", "2025-04-30"), ("2025Q2", "2025-07-31")])
spec = b.build()
res = Evaluator().evaluate_with_market(spec, MarketContext(), cutoff, cutoff)
print("Dependency tree for leverage:")
print(DependencyTracer(spec).dependency_tree_text("leverage"))

engine = CovenantEngine.from_template(CovenantTemplate.cov_lite(max_leverage=3.75, max_senior_leverage=3.25))
engine.validate()

node_mapping = {"total_leverage": "leverage", "senior_leverage": "leverage"}
terms = {
    "scope": "incurrence",
    "covenant_ids": ["max_total_leverage", "max_senior_leverage"],
    "observed_through": cutoff,
    "bindings": {metric: {"node": node, "unit": {"kind": "ratio"}} for metric, node in node_mapping.items()},
}
assessment = assess_statement_covenants(engine, spec, res, StatementCovenantSpec(periods=PERIODS, **terms))
compliance = assessment.to_pandas()
print(compliance[["date", "is_actual", "covenant_id", "state", "passed", "headroom_raw"]].to_string(index=False))
first = compliance.loc[compliance["covenant_id"] == "max_total_leverage"].iloc[0]
assert first["is_actual"] and abs(first["headroom_raw"] + 0.5) < 1e-12

forecast = forecast_statement_covenants(
    engine,
    spec,
    res,
    StatementCovenantSpec(periods=PERIODS[2:], **terms),
    CovenantForecastConfig(reference_date=cutoff).with_scope("incurrence"),
)
assert not forecast.assessment.to_pandas()["is_actual"].any()
print([(breach.covenant_id, breach.breach_date) for breach in forecast.breaches])

# Pass a single observation to the engine for the wire example later.
snapshot = {metric: res.get(node, PERIODS[-1]) for metric, node in node_mapping.items()}
report = engine.evaluate(snapshot, AS_OF_DATES[-1])
print(reports_table(report).to_pandas())

In [ ]:
print("Downside metric snapshots through the same covenant engine")
downside_snapshots = {
    "2026Q1": ("2026-03-31", 4.076),
    "2026Q2": ("2026-06-30", 3.940),
}
for period, (as_of, leverage) in downside_snapshots.items():
    report = engine.evaluate({"total_leverage": leverage, "senior_leverage": leverage}, as_of)
    print(period, {covenant_id: item.passed for covenant_id, item in report.items()})
print("All pass/fail decisions above come from CovenantEngine.evaluate.")

## Covenant engine presets (`finstack_quant.covenants`)

The `covenants` module ships standard packages as one `CovenantTemplate` per package — `cov_lite`, `lbo_standard`, `project_finance` and `real_estate` — expanded into typed specs by `covenant_template`; WASM's `covenantTemplate` takes the same template as its wire object. The model supplies tested metrics; `CovenantEngine.evaluate(metrics, as_of)` (and its JSON twin `evaluate_engine`) remains the single pass/fail and headroom implementation.

In [ ]:
from finstack_quant.covenants import (
    evaluate_engine,
    validate_covenant_report_json,
    validate_covenant_spec_json,
)

presets = {
    "cov_lite": covenant_template(CovenantTemplate.cov_lite(max_leverage=5.0, max_senior_leverage=3.5)),
    "lbo_standard": covenant_template(CovenantTemplate.lbo_standard(
        initial_leverage=6.0, interest_coverage=2.0, fixed_charge_coverage=1.1, max_capex=50.0
    )),
    "project_finance": covenant_template(CovenantTemplate.project_finance(
        min_dscr=1.2, distribution_lockup_dscr=1.1, min_liquidity=10.0, max_net_leverage=7.0
    )),
    "real_estate": covenant_template(CovenantTemplate.real_estate(min_dscr=1.25, min_debt_yield=0.08, max_ltv=0.75)),
}
for name, specs in presets.items():
    print(f"{name}: {len(specs)} covenants — {[spec.covenant.label for spec in specs]}")

# The JSON bridge is the same kernel: a freshly serialized engine evaluates
# identically through evaluate_engine (ignore in-memory series history).
fresh = CovenantEngine.from_specs(list(engine.specs))
direct = fresh.evaluate(snapshot, AS_OF_DATES[-1])
bridged = evaluate_engine(fresh.to_json(), snapshot, AS_OF_DATES[-1])
assert set(bridged) == set(direct)
for covenant_id, original in direct.items():
    bridged_report = bridged[covenant_id]
    assert bridged_report.passed == original.passed
    assert bridged_report.headroom == original.headroom
    assert bridged_report.headroom_raw == original.headroom_raw

validate_covenant_spec_json(engine.specs[0].to_json())
validate_covenant_report_json(direct[next(iter(direct))].to_json())
print("Preset spec and latest engine result validated OK")

## Takeaways

- Encode tested metrics as statement nodes and covenant definitions as engine specs.
- Route every base and downside compliance decision through `CovenantEngine.evaluate` / `evaluate_series`.
- Springing conditions, waivers, step-down schedules and incurrence scope are typed (`SpringingCondition`, `CovenantWaiver`, `ThresholdSchedule`, `Covenant.with_scope`); test windows stay in the engine JSON.